# 05 — A1 Phase-Aware U-Net Evaluation

This notebook evaluates A1 on the **same frozen VoiceBank+DEMAND test manifest used for A0**.

Outputs are written to:

`D:\PAPERS\SPEECH\low_snr_speech_enhancement\outputs\a1_phaseaware\evaluation`

Metrics:
- PESQ-WB
- STOI
- ESTOI
- SI-SDR
- reference-relative SNR

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from pesq import pesq
from pystoi import stoi

PROJECT_ROOT = Path(r"D:\PAPERS\SPEECH\low_snr_speech_enhancement")

SR = 16000
N_FFT = 512
WIN_LENGTH = 400
HOP_LENGTH = 100

CHECKPOINT = (
    PROJECT_ROOT /
    "outputs" /
    "a1_phaseaware" /
    "best.pt"
)

TEST_MANIFEST = (
    PROJECT_ROOT /
    "manifests" /
    "voicebank" /
    "test.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT /
    "outputs" /
    "a1_phaseaware" /
    "evaluation"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert CHECKPOINT.exists(), (
    f"Checkpoint missing: {CHECKPOINT}"
)
assert TEST_MANIFEST.exists(), (
    f"Test manifest missing: {TEST_MANIFEST}"
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)
print("Checkpoint:", CHECKPOINT)
print("Test manifest:", TEST_MANIFEST)

In [ ]:
def read_audio(path, target_sr=16000):
    wav, sr = sf.read(
        path,
        always_2d=False
    )

    if wav.ndim > 1:
        wav = wav.mean(axis=1)

    wav = np.asarray(
        wav,
        dtype=np.float32
    )

    if sr != target_sr:
        wav = librosa.resample(
            wav,
            orig_sr=sr,
            target_sr=target_sr,
            res_type="kaiser_best"
        ).astype(np.float32)

    return wav

def stft_complex(waveform):
    window = torch.hann_window(
        WIN_LENGTH,
        device=waveform.device,
        dtype=waveform.dtype
    )

    return torch.stft(
        waveform,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=window,
        center=True,
        return_complex=True
    )

def istft_complex(spec, length):
    window = torch.hann_window(
        WIN_LENGTH,
        device=spec.device,
        dtype=spec.real.dtype
    )

    return torch.istft(
        spec,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=window,
        center=True,
        length=length
    )

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(
                in_ch,
                out_ch,
                3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_ch,
                out_ch,
                3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

class PhaseAwareUNet(nn.Module):
    def __init__(
        self,
        base_channels=16,
        depth=4
    ):
        super().__init__()

        channels = [
            base_channels * (2 ** i)
            for i in range(depth)
        ]

        self.encoders = nn.ModuleList()
        in_ch = 1

        for ch in channels:
            self.encoders.append(
                ConvBlock(in_ch, ch)
            )
            in_ch = ch

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(
            channels[-1],
            channels[-1] * 2
        )

        self.up_convs = nn.ModuleList()
        self.decoders = nn.ModuleList()

        dec_in = channels[-1] * 2

        for ch in reversed(channels):
            self.up_convs.append(
                nn.Conv2d(
                    dec_in,
                    ch,
                    1
                )
            )

            self.decoders.append(
                ConvBlock(
                    ch * 2,
                    ch
                )
            )

            dec_in = ch

        self.mag_head = nn.Conv2d(
            channels[0],
            1,
            1
        )

        self.phase_head = nn.Conv2d(
            channels[0],
            2,
            1
        )

    def forward(self, noisy_spec):
        noisy_mag = noisy_spec.abs()
        noisy_phase = torch.angle(
            noisy_spec
        )

        x = torch.log1p(
            noisy_mag
        ).unsqueeze(1)

        skips = []

        for enc in self.encoders:
            x = enc(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for up, dec, skip in zip(
            self.up_convs,
            self.decoders,
            reversed(skips)
        ):
            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

            x = up(x)

            x = torch.cat(
                [x, skip],
                dim=1
            )

            x = dec(x)

        mag_mask = torch.sigmoid(
            self.mag_head(x)
        ).squeeze(1)

        enhanced_mag = (
            mag_mask * noisy_mag
        )

        phase_uv = self.phase_head(x)

        u = phase_uv[:, 0]
        v = phase_uv[:, 1]

        norm = torch.sqrt(
            u.square() +
            v.square() +
            1e-8
        )

        u = u / norm
        v = v / norm

        delta_phase = torch.atan2(
            v,
            u
        )

        enhanced_phase = (
            noisy_phase +
            delta_phase
        )

        enhanced_spec = torch.polar(
            enhanced_mag,
            enhanced_phase
        )

        return {
            "enhanced_spec": enhanced_spec,
            "enhanced_mag": enhanced_mag,
            "enhanced_phase": enhanced_phase,
            "mag_mask": mag_mask,
            "delta_phase": delta_phase
        }

model = PhaseAwareUNet(
    base_channels=16,
    depth=4
).to(device)

checkpoint = torch.load(
    CHECKPOINT,
    map_location=device
)

model.load_state_dict(
    checkpoint["model"]
)

model.eval()

print(
    "Loaded epoch:",
    checkpoint.get("epoch")
)

print(
    "Checkpoint validation loss:",
    checkpoint.get("val_total")
)

## Metrics

In [ ]:
def align(clean, estimate):
    n = min(
        len(clean),
        len(estimate)
    )

    return (
        np.asarray(
            clean[:n],
            dtype=np.float64
        ),
        np.asarray(
            estimate[:n],
            dtype=np.float64
        )
    )

def si_sdr(
    clean,
    estimate,
    eps=1e-8
):
    clean, estimate = align(
        clean,
        estimate
    )

    clean = clean - clean.mean()
    estimate = (
        estimate - estimate.mean()
    )

    scale = (
        np.dot(estimate, clean) /
        (
            np.dot(clean, clean) +
            eps
        )
    )

    target = scale * clean
    residual = estimate - target

    return 10.0 * np.log10(
        (
            np.sum(target ** 2) +
            eps
        ) /
        (
            np.sum(residual ** 2) +
            eps
        )
    )

def snr_ref(
    clean,
    estimate,
    eps=1e-8
):
    clean, estimate = align(
        clean,
        estimate
    )

    error = estimate - clean

    return 10.0 * np.log10(
        (
            np.sum(clean ** 2) +
            eps
        ) /
        (
            np.sum(error ** 2) +
            eps
        )
    )

def compute_metrics(
    clean,
    estimate
):
    clean, estimate = align(
        clean,
        estimate
    )

    result = {}

    try:
        result["pesq"] = float(
            pesq(
                SR,
                clean,
                estimate,
                "wb"
            )
        )
    except Exception:
        result["pesq"] = np.nan

    try:
        result["stoi"] = float(
            stoi(
                clean,
                estimate,
                SR,
                extended=False
            )
        )
    except Exception:
        result["stoi"] = np.nan

    try:
        result["estoi"] = float(
            stoi(
                clean,
                estimate,
                SR,
                extended=True
            )
        )
    except Exception:
        result["estoi"] = np.nan

    result["si_sdr"] = float(
        si_sdr(clean, estimate)
    )

    result["snr_ref"] = float(
        snr_ref(clean, estimate)
    )

    return result

## Evaluate A1 on the frozen test set

In [ ]:
test_df = pd.read_csv(
    TEST_MANIFEST
)

rows = []

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Evaluating A1"
):
    clean = read_audio(
        row["clean_path"],
        SR
    )

    noisy = read_audio(
        row["noisy_path"],
        SR
    )

    n = min(
        len(clean),
        len(noisy)
    )

    clean = clean[:n]
    noisy = noisy[:n]

    noisy_t = torch.from_numpy(
        noisy
    ).unsqueeze(0).to(device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        noisy_spec = stft_complex(
            noisy_t
        )

        outputs = model(
            noisy_spec
        )

        enhanced = istft_complex(
            outputs["enhanced_spec"],
            length=n
        )

        mean_abs_phase_correction = float(
            outputs["delta_phase"]
            .abs()
            .mean()
            .cpu()
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = (
        time.perf_counter() -
        start
    )

    enhanced = (
        enhanced
        .squeeze(0)
        .cpu()
        .numpy()
    )

    noisy_metrics = compute_metrics(
        clean,
        noisy
    )

    enhanced_metrics = compute_metrics(
        clean,
        enhanced
    )

    duration_sec = n / SR
    rtf = elapsed / max(
        duration_sec,
        1e-8
    )

    rows.append({
        "utt_id": row["utt_id"],
        "speaker_id": row.get(
            "speaker_id",
            None
        ),
        **{
            f"noisy_{k}": v
            for k, v
            in noisy_metrics.items()
        },
        **{
            f"enh_{k}": v
            for k, v
            in enhanced_metrics.items()
        },
        "mean_abs_phase_correction_rad":
            mean_abs_phase_correction,
        "inference_time_sec": elapsed,
        "duration_sec": duration_sec,
        "rtf": rtf
    })

result = pd.DataFrame(rows)

per_utt_csv = (
    OUTPUT_DIR /
    "per_utterance.csv"
)

result.to_csv(
    per_utt_csv,
    index=False
)

print("Saved:", per_utt_csv)
display(result.head())

if torch.cuda.is_available():
    print(
        "Peak evaluation GPU memory (MB):",
        torch.cuda.max_memory_allocated() /
        (1024**2)
    )

## Aggregate A1 results

In [ ]:
metric_cols = [
    "noisy_pesq",
    "enh_pesq",
    "noisy_stoi",
    "enh_stoi",
    "noisy_estoi",
    "enh_estoi",
    "noisy_si_sdr",
    "enh_si_sdr",
    "noisy_snr_ref",
    "enh_snr_ref",
    "mean_abs_phase_correction_rad",
    "rtf"
]

summary_rows = []

for col in metric_cols:
    values = pd.to_numeric(
        result[col],
        errors="coerce"
    ).dropna()

    summary_rows.append({
        "metric": col,
        "mean": values.mean(),
        "std": values.std(ddof=1),
        "median": values.median(),
        "n": len(values)
    })

summary_df = pd.DataFrame(
    summary_rows
)

summary_csv = (
    OUTPUT_DIR /
    "summary.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False
)

display(summary_df)

print(
    "Mean RTF:",
    result["rtf"].mean()
)

print(
    "Median RTF:",
    result["rtf"].median()
)